# 五个新增随机日期：沪深股票和 ETF 全量验证
从无既往 *-run.json 日期记录且五类 raw parquet 文件齐全的日期中，固定种子 20260909 抽五日。沿用已通过十日回归的冻结二进制，6 进程并发，标准窗口，不做自动修复。

In [ ]:
import json, random
from pathlib import Path
root = Path.cwd() if (Path.cwd() / 'Cargo.toml').exists() else Path.cwd().parent
out = root / 'reports/20260908-additional-random-full'
manifest = json.loads((out / 'manifest.json').read_text())
assert sorted(random.Random(manifest['seed']).sample(manifest['eligible_random_dates'], manifest['random_count'])) == manifest['random_days']
assert not set(manifest['random_days']) & set(manifest['previously_tested_dates'])
print(manifest['status'], manifest['heartbeat_at'], manifest['random_days'])

In [ ]:
summary = json.loads((out / 'summary.json').read_text())
for r in summary['rows']:
    c, t = r.get('counts', {}), r.get('timing_summary', {})
    print(r['date'], r['market'], r['status'], c.get('matched'), c.get('mismatched'), c.get('excluded_by_status'), c.get('data_errors'), c.get('missing_source'), t.get('restore_total_seconds'), t.get('validation_total_seconds'), r.get('elapsed_seconds'), r.get('peak_rss_kib'))
    if r.get('first_anomalies') or r.get('error'):
        print(r.get('first_anomalies'), r.get('error'))

## 审计和计时口径
可比匹配率与停牌排除覆盖分别报告。全部失败明细在逐日 full.json，不把尚未完成任务算通过。恢复 = 输入分片 + 排除验证回调的回放 + 清理；对比 = 参考帧读取分类 + 比较回调 + 报告整理。总时间另含启动和输出。六进程并发的 instrumented wall time 不等于纯 CPU 时间或独立回放基准，GNU time 给出单进程峰值 RSS。